In [10]:
%pip install gdown
%pip install -q ultralytics==8.3.0 "opencv-python>=4.10" pillow matplotlib
%pip install -q "onnx>=1.16" "onnxruntime>=1.18" "onnxscript>=0.1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.0/683.0 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 13.3 MB/s eta 0:00:00


In [12]:
import os
import zipfile
import gdown
import shutil
import torch, platform
from ultralytics import YOLO

In [3]:
data_dir = os.path.join(os.getcwd(), "data")

def descargar_dataset(data_dir):

  # Verificar si exsite la carpeta
  if not os.path.exists(data_dir):

    try:
      # Nombre del archivo ZIP que se va a guardar
      ZIP_NAME = "kaggle-xray_baggage_scanner_anomaly_detection.zip"
      zip_path = os.path.join(data_dir, ZIP_NAME)

      # Crear carpeta data
      os.makedirs(data_dir, exist_ok=True)

      # Descargar el ZIP
      print("Descargando archivo zip ...")
      # Lo tomamos de Google Drive porque Kaggle requeire autenticación
      gdown.download(id="1IqPblTm7nmKFpHXtl4beopE_SajTBoI0", output=zip_path, quiet=False)
      print("Archivo zip descargado.")

      # Descomprimir el ZIP
      print("Descomprimiendo archivo ...")
      with zipfile.ZipFile(zip_path, "r") as zf:
          zf.extractall(data_dir)
      print("Archivo descomprimido.")
    except:
      # En caso de error, eliminar la carpeta creada
      shutil.rmtree(data_dir, ignore_errors=True)
      print("Ocurrió un error al descargar el dataset.")

  print(f"Dataset descargado en: '{data_dir}'")


# Descargar el dataset (solo si no existe la carpeta data)
descargar_dataset(data_dir)

Dataset descargado en: '/content/data'


In [4]:
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.12
PyTorch: 2.9.0+cu126
CUDA disponible: True
GPU: NVIDIA A100-SXM4-40GB


In [5]:
DATA_YAML = os.path.join(data_dir, "data.yaml")
BASE_WEIGHTS = "yolov8s.pt" if torch.cuda.is_available() else "yolov8n.pt"
IMG_SIZE = 416
EPOCHS   = 100

DEVICE = 0 if torch.cuda.is_available() else "cpu"  # autodetección
print(f"DATA: {DATA_YAML} | WEIGHTS: {BASE_WEIGHTS} | DEVICE: {DEVICE}")


DATA: /content/data/data.yaml | WEIGHTS: yolov8s.pt | DEVICE: 0


In [6]:
# ============================================
# 3) Entrenamiento (equivalente a CLI de README)
# ============================================
model = YOLO(BASE_WEIGHTS)

train_results = model.train(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    device=DEVICE,     # 0 en GPU, "cpu" si no hay GPU
    project="runs_yolo",
    name=f"y8n_{IMG_SIZE}e{EPOCHS}",
    cache=True,        # acelera IO en Colab
    verbose=True
)

best_path = model.trainer.best
print("Best weights:", best_path)

New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/data/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=416, save=True, save_period=-1, cache=True, device=0, workers=8, project=runs_yolo, name=y8n_416e100, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=F

100%|██████████| 755k/755k [00:00<00:00, 18.7MB/s]


Overriding model.yaml nc=80 with nc=5

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


...


100 epochs completed in 1.069 hours.
Optimizer stripped from runs_yolo/y8n_416e100/weights/last.pt, 19.9MB
Optimizer stripped from runs_yolo/y8n_416e100/weights/best.pt, 19.9MB

Validating runs_yolo/y8n_416e100/weights/best.pt...
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 186 layers, 9,829,599 parameters, 0 gradients, 23.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 56/56 [00:07<00:00,  7.78it/s]


                   all       1766       1766      0.841      0.801      0.827      0.363
                     0        391        391      0.971       0.98      0.975      0.505
                     1        389        389      0.829      0.846      0.862      0.361
                     2        225        225      0.752      0.511      0.607      0.217
                     3        366        366      0.769      0.773      0.791      0.353
                     4        395        395      0.886      0.894      0.897      0.378
Speed: 0.1ms preprocess, 0.5ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs_yolo/y8n_416e100


lr/pg0,▃▆███▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁
lr/pg1,▃████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁
lr/pg2,▃████▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
metrics/mAP50(B),▁▃▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇█▇████████████████████
metrics/mAP50-95(B),▁▁▃▃▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇████████████████
metrics/precision(B),▃▂▄▄▁▂▅▆▅▅▅▇▆▆▆▇▆▇▇▇▇█▇▇▇▇█▇████▇█▇█████
metrics/recall(B),▁▂▂▃▄▄▄▄▅▆▆▅▆▆▇▆▆▇▇▇▇▇▇▇▇▇▇██████▇█▇████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


Best weights: runs_yolo/y8n_416e100/weights/best.pt


In [7]:
# ============================================
# 4) Validación (val y test si existen en data.yaml)
# ============================================
metrics_val = model.val(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    device=DEVICE,
    split="val"
)
print("VAL metrics:", metrics_val)

Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 186 layers, 9,829,599 parameters, 0 gradients, 23.4 GFLOPs


val: Scanning /content/data/valid/labels.cache... 1766 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1766/1766 [00:00<?, ?it/s]

WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.



val: Caching images (0.9GB RAM): 100%|██████████| 1766/1766 [00:00<00:00, 5679.36it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 111/111 [00:07<00:00, 14.05it/s]


                   all       1766       1766      0.841        0.8      0.826      0.362
                     0        391        391      0.971       0.98      0.975      0.505
                     1        389        389      0.831      0.848       0.87      0.361
                     2        225        225      0.753      0.511      0.606      0.218
                     3        366        366      0.766      0.768      0.783      0.351
                     4        395        395      0.886      0.894      0.897      0.376
Speed: 0.1ms preprocess, 0.8ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs_yolo/y8n_416e1002
VAL metrics: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a53a21b3710>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(

In [8]:
metrics_test = model.val(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    device=DEVICE,
    split="test"
)
print("TEST metrics:", metrics_test)

Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)


val: Scanning /content/data/test/labels... 883 images, 0 backgrounds, 0 corrupt: 100%|██████████| 883/883 [00:00<00:00, 1524.35it/s]

val: New cache created: /content/data/test/labels.cache


WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


val: Caching images (0.4GB RAM): 100%|██████████| 883/883 [00:00<00:00, 5940.94it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 56/56 [00:04<00:00, 12.98it/s]


                   all        883        883      0.894      0.806      0.868       0.39
                     0        166        166      0.953      0.958      0.978      0.521
                     1        193        193       0.85      0.829      0.883      0.392
                     2        118        118      0.882      0.551      0.698      0.276
                     3        203        203       0.86      0.808      0.848      0.378
                     4        203        203      0.926      0.887      0.932      0.381
Speed: 0.1ms preprocess, 0.8ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs_yolo/y8n_416e1003
TEST metrics: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a52ec129160>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence

In [13]:
# ============================================
# 6) Exportar mejor modelo (opcional)
# ============================================
best_model = YOLO(best_path)
best_model.export(format="onnx", imgsz=IMG_SIZE)
print("Export ONNX listo.")

Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 186 layers, 9,829,599 parameters, 0 gradients, 23.4 GFLOPs

PyTorch: starting from 'runs_yolo/y8n_416e100/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 9, 3549) (19.0 MB)

ONNX: starting export with onnx 1.19.1 opset 10...


W1123 00:58:15.311000 4085 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 10 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter

Applied 1 of general pattern rewrite rules.
ONNX: slimming with onnxslim 0.1.34...
ONNX: export success ✅ 3.3s, saved as 'runs_yolo/y8n_416e100/weights/best.onnx' (37.6 MB)

Export complete (3.7s)
Results saved to /content/runs_yolo/y8n_416e100/weights
Predict:         yolo predict task=detect model=runs_yolo/y8n_416e100/weights/best.onnx imgsz=416  
Validate:        yolo val task=detect model=runs_yolo/y8n_416e100/weights/best.onnx imgsz=416 data=/content/data/data.yaml  
Visualize:       https://netron.app
Export ONNX listo.
